# Signage & Wayfinding — Embedding Generator

This notebook:
1. Takes your comments CSV
2. Generates text embeddings via OpenAI
3. Pushes the result straight to your GitHub Pages site

**Run each cell top to bottom. You will be prompted for your API key, GitHub token, and CSV file.**

In [ ]:
# Install dependencies
!pip install openai numpy requests -q

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Your GitHub repo details
GITHUB_OWNER = "tinydogboots"
GITHUB_REPO  = "apdxvc"
GITHUB_BRANCH = "gh-pages"

# Output filenames (will be placed in data/ in the repo)
OUTPUT_NAME = "comments"   # → data/comments.bytes + data/comments_metadata.tsv
TENSOR_NAME = "Signage & Wayfinding Comments"  # label shown in the projector

# Embedding model
EMBED_MODEL = "text-embedding-3-small"
# ────────────────────────────────────────────────────────────────────────────────
print("Config set.")

In [ ]:
import getpass

OPENAI_KEY  = getpass.getpass("Paste your OpenAI API key: ")
GITHUB_TOKEN = getpass.getpass("Paste your GitHub personal access token (needs repo scope): ")
print("Credentials stored in memory only.")

**Need a GitHub token?**
1. Go to github.com → Settings → Developer settings → Personal access tokens → Tokens (classic)
2. Click **Generate new token (classic)**
3. Check the **repo** scope
4. Copy and paste it above

In [ ]:
# Upload your CSV file
from google.colab import files
import io, csv

print("Select your comments CSV file...")
uploaded = files.upload()
filename = list(uploaded.keys())[0]
raw = uploaded[filename].decode("utf-8")
print(f"Loaded: {filename}")

reader = csv.DictReader(io.StringIO(raw))
rows = list(reader)
columns = reader.fieldnames
print(f"{len(rows)} rows, columns: {columns}")

In [ ]:
# ── Tell us which column holds the comment text ────────────────────────────────
# Look at the printed column names above and set this:
COMMENT_COLUMN = "comment"   # ← change to match your CSV header

# All other columns will become metadata (shown on hover in the projector)
# ────────────────────────────────────────────────────────────────────────────────

texts = [r[COMMENT_COLUMN] for r in rows]
print(f"First comment: {texts[0][:120]}")

In [ ]:
import numpy as np
from openai import OpenAI

client = OpenAI(api_key=OPENAI_KEY)

BATCH = 100
all_vectors = []

for i in range(0, len(texts), BATCH):
    batch = texts[i:i+BATCH]
    resp = client.embeddings.create(model=EMBED_MODEL, input=batch)
    all_vectors.extend([d.embedding for d in resp.data])
    print(f"Embedded {min(i+BATCH, len(texts))}/{len(texts)}")

vectors = np.array(all_vectors, dtype=np.float32)
print(f"Vectors shape: {vectors.shape}")

In [ ]:
import base64, json

# Build .bytes content (raw float32, little-endian)
bytes_content = vectors.astype("<f4").tobytes()
bytes_b64 = base64.b64encode(bytes_content).decode()

# Build metadata TSV
meta_cols = [c for c in columns if c != COMMENT_COLUMN]
tsv_lines = [COMMENT_COLUMN + ("\t" + "\t".join(meta_cols) if meta_cols else "")]
for r in rows:
    line = r[COMMENT_COLUMN]
    if meta_cols:
        line += "\t" + "\t".join(str(r.get(c, "")) for c in meta_cols)
    tsv_lines.append(line)
tsv_content = "\n".join(tsv_lines)
tsv_b64 = base64.b64encode(tsv_content.encode()).decode()

print("Files ready.")
print(f"  .bytes : {len(bytes_content):,} bytes")
print(f"  .tsv   : {len(tsv_content):,} chars, {len(tsv_lines)-1} rows")

In [ ]:
import requests

HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json",
}
API = f"https://api.github.com/repos/{GITHUB_OWNER}/{GITHUB_REPO}/contents"

def push_file(path, b64_content, message):
    url = f"{API}/{path}"
    # Get current SHA if file exists (required for updates)
    r = requests.get(url, headers={**HEADERS, "ref": GITHUB_BRANCH})
    sha = r.json().get("sha") if r.status_code == 200 else None
    payload = {"message": message, "content": b64_content, "branch": GITHUB_BRANCH}
    if sha:
        payload["sha"] = sha
    r = requests.put(url, headers=HEADERS, json=payload)
    r.raise_for_status()
    print(f"  ✓ {path}")

bytes_path = f"data/{OUTPUT_NAME}.bytes"
tsv_path   = f"data/{OUTPUT_NAME}_metadata.tsv"

print("Pushing to GitHub...")
push_file(bytes_path, bytes_b64,   f"Add {OUTPUT_NAME} embeddings (.bytes)")
push_file(tsv_path,   tsv_b64,     f"Add {OUTPUT_NAME} metadata (.tsv)")
print("Done.")

In [ ]:
# Fetch current projector_config.json and add/replace this embedding
config_path = "data/projector_config.json"
url = f"{API}/{config_path}"
r = requests.get(url, headers={**HEADERS}, params={"ref": GITHUB_BRANCH})
r.raise_for_status()
config_data = r.json()
config = json.loads(base64.b64decode(config_data["content"]).decode())
sha = config_data["sha"]

entry = {
    "tensorName": TENSOR_NAME,
    "tensorShape": list(vectors.shape),
    "tensorPath": bytes_path,
    "metadataPath": tsv_path,
}
config["embeddings"] = [e for e in config["embeddings"] if e.get("tensorName") != TENSOR_NAME]
config["embeddings"].insert(0, entry)

new_b64 = base64.b64encode(json.dumps(config, indent=2).encode()).decode()
r = requests.put(url, headers=HEADERS, json={
    "message": f"Add {TENSOR_NAME} to projector config",
    "content": new_b64,
    "sha": sha,
    "branch": GITHUB_BRANCH,
})
r.raise_for_status()
print("Updated projector_config.json")
print(f"\n✅ All done! Open your projector at:")
print(f"   https://{GITHUB_OWNER}.github.io/{GITHUB_REPO}/")